# Team Challenge · Sprint 03-04 - Catálogo de películas

**Dataset:**  
- [movies.csv](./data/movies.csv)
- [ratings.csv](./data/ratings.csv)
- [tags.csv](./data/tags.csv)
- [links.csv](./data/links.csv) 

**Entregables:** Repositorio de GitHub con el código fuente. Puede ser scripts de python o notebook ordenado, código reproducible, README con cómo ejecutar el proyecto.

**Control de versiones:** gestión del proyecto con **GitHub desde el primer día**. Trabajar en equipo en paralelo con **ramas**, **pull requests** y revisión entre compañeros.

**Parte 1 (Pandas):** Data analytics (Pandas)

**Parte 2 (TMDB):** una petición `GET /3/movie/{id}` por película; definid `TMDB_API_KEY` o `TMDB_READ_ACCESS_TOKEN` en el entorno (sin subir claves al repo).

**Parte 3 (Gemini):** añadir `overview_es` a `movies10` a partir de los `overview` de TMDB; definid `GEMINI_API_KEY` o usad `getpass` (sin subir claves al repo). Requiere `pip install google-genai`.

**Parte 4 (opcional):**
- **A)** catálogo ampliado (`pitch_es`, `edad_sugerida`, `temas`).
- **B)** recomendador por género.

## Trabajo en equipo y control de versiones

- Crear el **repositorio en GitHub** al inicio (día 1–2), no al final.
- Definir una rama principal (`main`) y una para desarrollo (`develop`) y ramas por tarea (p. ej. `feature/parte1-pandas`, `feature/tmdb`, `feature/gemini`).
- Integrar el trabajo con **pull requests** hacia `develop`; al menos **una PR revisada y mergeada por miembro** del equipo.
- Evitar trabajar en en `main` es la rama de "producción"
- Resolver conflictos en ramas antes del merge.
- **README:** cómo clonar el repo, instalar dependencias, configurar claves (`TMDB_*`, `GEMINI_API_KEY`) y ejecutar el notebook o scripts.
- **No subir claves** al repositorio (usar `.gitignore` para `.env` si aplica).

## Parte 1: Data analytics (Pandas)

### 1. Ingesta de datos

- Los CSV están en **`Team_Challenges/TC_01_Sprint_03_04/data/`**.
- Cargar con **Pandas** los cuatro ficheros: `movies.csv`, `ratings.csv`, `tags.csv`, `links.csv`.
- Comprobar para cada `DataFrame`: `shape`, columnas, `dtypes`, `head` y conteo de nulos.

In [2]:
import pandas as pd

# Aseguraos de que vuestra ruta desde el repo del trabajo sea correcta para poder cargar estos archivos.
movies = pd.read_csv("Team_Challenges/TC_01_Sprint_03_04/data/movies.csv")
ratings = pd.read_csv("Team_Challenges/TC_01_Sprint_03_04/data/ratings.csv")
tags = pd.read_csv("Team_Challenges/TC_01_Sprint_03_04/data/tags.csv")
links = pd.read_csv("Team_Challenges/TC_01_Sprint_03_04/data/links.csv")


In [3]:
print("SHAPE")
print(movies.shape)

print("\nCOLUMNAS")
print(movies.columns)

print("\nDTYPES")
print(movies.dtypes)

print("\nHEAD")
print(movies.head())

print("\nNULS")
print(movies.isnull().sum())

SHAPE
(9742, 3)

COLUMNAS
Index(['movieId', 'title', 'genres'], dtype='str')

DTYPES
movieId    int64
title        str
genres       str
dtype: object

HEAD
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

NULS
movieId    0
title      0
genres     0
dtype: int64


### 2. Columna `year` desde el título

- En el dataset de películas. el año de estreno suele aparecer **entre paréntesis al final** de `title`, p. ej. `Batman (1989)`.
- Implementad una función (`year_from_title`) que devuelva un entero de cuatro cifras o valor ausente (`NaN` / `<NA>`) si el título no sigue ese patrón.
- Añadid la columna **`year`** a `movies` como numérico (`pd.to_numeric(..., errors="coerce")`).
- Editar el campo `title` para que no contenga el año de estreno.
- Contad cuántas películas **no** tienen año reconocible y mostrad **una pequeña muestra** de sus títulos (casos límite).

In [4]:
import re

def year_from_title(title):
    match = re.search(r"\((\d{4})\)$", title)
    if match:
        return int(match.group(1))
    return pd.NA

# Crear columna year
movies["year"] = movies["title"].apply(year_from_title)
movies["year"] = pd.to_numeric(movies["year"], errors="coerce")

# Eliminar el año del título
movies["title"] = movies["title"].str.replace(r"\s*\(\d{4}\)$", "", regex=True)

# Contar películas sin año
num_sin_year = movies["year"].isna().sum()
print(f"Películas sin año reconocible: {num_sin_year}")

# Mostrar una pequeña muestra
print("\nMuestra de títulos sin año:")
print(movies.loc[movies["year"].isna(), "title"].head())

Películas sin año reconocible: 24

Muestra de títulos sin año:
5609    From Dusk Till Dawn 2: Texas Blood Money (1999) 
6059                                           Babylon 5
6706            Justice League: The New Frontier (2008) 
6718                       Assembly (Ji jie hao) (2007) 
7878                                  96 Minutes (2011) 
Name: title, dtype: str


### 2. Unificación de datos (*merge* / *join*)

- Entender las **claves** entre tablas (p. ej. `movieId` une `movies`, `ratings`, `tags` y `links`).
- Construir un esquema unificado: al menos una tabla **película–usuario–rating** (ratings enriquecida con título y géneros) y otra con **película–tags** si procede.
- Usar `merge` (u operaciones equivalentes) con criterio claro: tipo de unión (*inner* / *left*), duplicados generados y cómo se resuelven.
- Imprimir las 10 primeras filas de las tablas resultantes.
- Dejar documentado qué filas se pierden o se multiplican al unir y **por qué** es aceptable en vuestro caso de uso.

In [5]:

# 1) Tabla película–usuario–rating
# Usamos Merge how = left para conservar todas las valoraciones aunque alguna película no tenga todos los datos.
ratings_enriched = ratings.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left",
    validate="many_to_one"
)

print("ratings original:", ratings.shape)
print("ratings_enriched:", ratings_enriched.shape)
print("Ratings sin película encontrada:", ratings_enriched["title"].isna().sum())

print("\nPrimeras 10 filas de ratings_enriched:")
display(ratings_enriched.head(10))


# 2) Tabla película–tags
# Usamos LEFT para conservar todos los tags aunque a alguna película le falten datos.
movie_tags = tags.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left",
    validate="many_to_one"
)

print("\ntags original:", tags.shape)
print("movie_tags:", movie_tags.shape)
print("Tags sin película encontrada:", movie_tags["title"].isna().sum())

print("\nPrimeras 10 filas de movie_tags:")
display(movie_tags.head(10))


# 3) Comprobación de duplicados / multiplicación
print("\nDuplicados por clave en movies:")
print(movies["movieId"].duplicated().sum())

print("\nDuplicados exactos en ratings_enriched:")
print(ratings_enriched.duplicated().sum())

print("\nDuplicados exactos en movie_tags:")
print(movie_tags.duplicated().sum())


# 4) Documentación breve
print("""
DOCUMENTACIÓN DE UNIONES

ratings_enriched:
- Unión usada: LEFT JOIN desde ratings hacia movies por movieId. porque el caso de uso parte de las valoraciones; queremos conservar todas las filas de ratings.
- Pérdida de filas: no debería haber pérdida porque LEFT conserva todas las filas de ratings.
- Multiplicación: no debería multiplicar filas si movieId es único en movies.
- Validación usada: many_to_one, porque muchas valoraciones pueden apuntar a una misma película.

movie_tags:
- Unión usada: LEFT JOIN desde tags hacia movies por movieId porque también queremos conservar todos los tags disponibles.
- Pérdida de filas: no debería haber pérdida porque LEFT conserva todas las filas de tags.
- Multiplicación: no debería multiplicar filas si movieId es único en movies.
- Validación usada: many_to_one, porque muchos tags pueden apuntar a una misma película.

Es aceptable porque movies actúa como tabla de dimensión/datos, mientras ratings y tags son tablas válidas.
""")

ratings original: (100836, 4)
ratings_enriched: (100836, 6)
Ratings sin película encontrada: 0

Primeras 10 filas de ratings_enriched:


,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story,Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men,Comedy|Romance
2,1,6,4.0,964982224,Heat,Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The",Crime|Mystery|Thriller
5,1,70,3.0,964982400,From Dusk Till Dawn,Action|Comedy|Horror|Thriller
6,1,101,5.0,964980868,Bottle Rocket,Adventure|Comedy|Crime|Romance
7,1,110,4.0,964982176,Braveheart,Action|Drama|War
8,1,151,5.0,964984041,Rob Roy,Action|Drama|Romance|War
9,1,157,5.0,964984100,Canadian Bacon,Comedy|War



tags original: (3683, 4)
movie_tags: (3683, 6)
Tags sin película encontrada: 0

Primeras 10 filas de movie_tags:


,userId,movieId,tag,timestamp,title,genres
0,2,60756,funny,1445714994,Step Brothers,Comedy
1,2,60756,Highly quotable,1445714996,Step Brothers,Comedy
2,2,60756,will ferrell,1445714992,Step Brothers,Comedy
3,2,89774,Boxing story,1445715207,Warrior,Drama
4,2,89774,MMA,1445715200,Warrior,Drama
5,2,89774,Tom Hardy,1445715205,Warrior,Drama
6,2,106782,drugs,1445715054,"Wolf of Wall Street, The",Comedy|Crime|Drama
7,2,106782,Leonardo DiCaprio,1445715051,"Wolf of Wall Street, The",Comedy|Crime|Drama
8,2,106782,Martin Scorsese,1445715056,"Wolf of Wall Street, The",Comedy|Crime|Drama
9,7,48516,way too long,1169687325,"Departed, The",Crime|Drama|Thriller



Duplicados por clave en movies:
0

Duplicados exactos en ratings_enriched:
0

Duplicados exactos en movie_tags:
0

DOCUMENTACIÓN DE UNIONES

ratings_enriched:
- Unión usada: LEFT JOIN desde ratings hacia movies por movieId. porque el caso de uso parte de las valoraciones; queremos conservar todas las filas de ratings.
- Pérdida de filas: no debería haber pérdida porque LEFT conserva todas las filas de ratings.
- Multiplicación: no debería multiplicar filas si movieId es único en movies.
- Validación usada: many_to_one, porque muchas valoraciones pueden apuntar a una misma película.

movie_tags:
- Unión usada: LEFT JOIN desde tags hacia movies por movieId porque también queremos conservar todos los tags disponibles.
- Pérdida de filas: no debería haber pérdida porque LEFT conserva todas las filas de tags.
- Multiplicación: no debería multiplicar filas si movieId es único en movies.
- Validación usada: many_to_one, porque muchos tags pueden apuntar a una misma película.

Es aceptable po

### 4. Agregaciones y segmentación

- Usar `groupby` (por usuario, por película o por género) con al menos una agregación multi-columna (`agg`).
- Comparar dos segmentos (p. ej. usuarios con muchas valoraciones vs. pocos; o por **década** usando la columna `year` del apartado 3) con tablas resumen.

In [6]:

# GROUPBY + AGG
# Resumen por usuario
user_summary = ratings_enriched.groupby("userId").agg(
    num_ratings=("rating", "count"),
    media_rating=("rating", "mean"),
    min_rating=("rating", "min"),
    max_rating=("rating", "max")
)

print("Resumen por usuario:")
display(user_summary.head(10))


# Comparación de segmentos

# Usuarios con muchas vs pocas valoraciones

umbral = user_summary["num_ratings"].median()

muchas = user_summary[user_summary["num_ratings"] >= umbral]
pocas = user_summary[user_summary["num_ratings"] < umbral]

comparacion = pd.DataFrame({
    "Grupo": ["Muchas valoraciones", "Pocas valoraciones"],
    "Usuarios": [len(muchas), len(pocas)],
    "Media ratings por usuario": [
        muchas["num_ratings"].mean(),
        pocas["num_ratings"].mean()
    ],
    "Media puntuación": [
        muchas["media_rating"].mean(),
        pocas["media_rating"].mean()
    ]
})

print("\nComparación de segmentos:")
display(comparacion)


Resumen por usuario:


,num_ratings,media_rating,min_rating,max_rating
userId,,,,
1,232,4.366379,1.0,5.0
2,29,3.948276,2.0,5.0
3,39,2.435897,0.5,5.0
4,216,3.555556,1.0,5.0
5,44,3.636364,1.0,5.0
6,314,3.493631,1.0,5.0
7,152,3.230263,0.5,5.0
8,47,3.574468,1.0,5.0
9,46,3.260870,1.0,5.0



Comparación de segmentos:


,Grupo,Usuarios,Media ratings por usuario,Media puntuación
0,Muchas valoraciones,305,292.226230,3.607785
1,Pocas valoraciones,305,38.383607,3.706660


### 5. Preguntas sobre los datos (`movies`)

**Requisito:** haber creado la columna `year` en el **apartado 2**.

1. **Cuántas** películas están listadas en `movies`.
2. **Cuáles** son las **más antiguas** (menor año extraído del título).
3. **Cuántas** tienen **"Dracula"** en el título (coincidencia parcial, sin distinguir mayúsculas).
4. **Títulos más comunes**.
5. Películas con **"Exorcist"** ordenadas de la más antigua a la más moderna.
6. **Cuántas** con año **1950**.
7. **Cuántas** entre **1950 y 1959** inclusive.
8. **Año** de la película con título exacto **`Batman`** (y contraste con otras *Batman*).
9. Listado de películas que tienen como tag "sci-fi" y "adventure"
10. ¿Cuál es la tag más repetida?

In [7]:
# 1. Cuántas películas hay
print("1. Películas en movies:", len(movies))


# 2. Películas más antiguas
min_year = movies["year"].min()
print("\n2. Películas más antiguas:")
display(movies[movies["year"] == min_year][["title", "year"]])


# 3. Cuántas tienen Dracula en el título
dracula = movies[movies["title"].str.contains("Dracula", case=False, na=False)]
print("\n3. Películas con Dracula:", len(dracula))


# 4. Títulos más comunes
print("\n4. Títulos más comunes:")
display(movies["title"].value_counts().head(10))


# 5. Películas con Exorcist ordenadas por año
exorcist = movies[movies["title"].str.contains("Exorcist", case=False, na=False)]
print("\n5. Películas con Exorcist:")
display(exorcist.sort_values("year")[["title", "year", "genres"]])


# 6. Cuántas con año 1950
print("\n6. Películas de 1950:", (movies["year"] == 1950).sum())


# 7. Cuántas entre 1950 y 1959 inclusive
print("\n7. Películas entre 1950 y 1959:", movies["year"].between(1950, 1959).sum())


# 8. Año de Batman exacto y contraste con otras Batman
batman_exact = movies[movies["title"].eq("Batman")]
batman_all = movies[movies["title"].str.contains("Batman", case=False, na=False)]

print("\n8. Batman exacto:")
display(batman_exact[["title", "year", "genres"]])

print("\nOtras Batman:")
display(batman_all.sort_values("year")[["title", "year", "genres"]])


# 9. Películas que tienen como tag sci-fi y adventure
tags_lower = tags.copy()
tags_lower["tag"] = tags_lower["tag"].str.lower()

movie_tags_filtered = tags_lower[tags_lower["tag"].isin(["sci-fi", "adventure"])]

movies_sci_fi_adventure = (
    movie_tags_filtered
    .groupby("movieId")["tag"]
    .apply(set)
    .reset_index()
)

movies_sci_fi_adventure = movies_sci_fi_adventure[
    movies_sci_fi_adventure["tag"].apply(lambda x: {"sci-fi", "adventure"}.issubset(x))
]

movies_sci_fi_adventure = movies_sci_fi_adventure.merge(
    movies[["movieId", "title", "year", "genres"]],
    on="movieId",
    how="left"
)

print("\n9. Películas con tags sci-fi y adventure:")
display(movies_sci_fi_adventure[["title", "year", "genres", "tag"]])


# 10. Tag más repetida
tag_counts = tags_lower["tag"].value_counts()

print("\n10. Tag más repetida:")
print(tag_counts.head(1))

1. Películas en movies: 9742

2. Películas más antiguas:


,title,year
5868,"Trip to the Moon, A (Voyage dans la lune, Le)",1902.0



3. Películas con Dracula: 9

4. Títulos más comunes:


title
Hamlet                          5
Misérables, Les                 4
Three Musketeers, The           4
Jane Eyre                       4
Christmas Carol, A              4
Little Women                    3
Hunchback of Notre Dame, The    3
Emma                            3
Spellbound                      3
Cinderella                      3
Name: count, dtype: int64


5. Películas con Exorcist:


...


Otras Batman:


,title,year,genres
5463,Batman,1966.0,Action|Adventure|Comedy
509,Batman,1989.0,Action|Crime|Thriller
1060,Batman Returns,1992.0,Action|Crime
2418,Batman: Mask of the Phantasm,1993.0,Animation|Children
126,Batman Forever,1995.0,Action|Adventure|Comedy|Crime
1174,Batman & Robin,1997.0,Action|Adventure|Fantasy|Thriller
5620,"Batman/Superman Movie, The",1998.0,Action|Adventure|Animation|Children|Fantasy|Sc...
5631,Batman Beyond: Return of the Joker,2000.0,Action|Animation|Crime|Sci-Fi|Thriller
8234,Batman: Mystery of the Batwoman,2003.0,Action|Animation|Children|Crime
5917,Batman Begins,2005.0,Action|Crime|IMAX



9. Películas con tags sci-fi y adventure:


,title,year,genres,tag



10. Tag más repetida:
tag
in netflix queue    131
Name: count, dtype: int64


## Parte 2: Petición HTTP a la API de TMDB

### Endpoint

`GET https://api.themoviedb.org/3/movie/{tmdb_id}`

Ejemplo público de la misma forma que en la documentación de TMDB: **`https://api.themoviedb.org/3/movie/100`** (el número es el `tmdb_id`; en vuestro caso usaréis el `tmdbId` de cada fila de `links`).

### Tareas

1. Construir un **`DataFrame` `movies10`** con **10 películas** del dataset original (por ejemplo las 10 primeras filas que tengan `tmdbId` tras unir `movies` con `links`).
2. Escribir una función **`fetch_movie_details(tmdb_id)`** que haga la petición anterior y devuelva al menos **`overview`** y **`homepage`** (texto vacío si vienen nulos).
3. Recorrer `movies10` y **añadir** a cada registro esas dos columnas en el propio `DataFrame`.

### Autenticación

Para poder acceder a la API de TMDB, debéis hacer uso del ACCESS TOKEN o de la API KEY que te proporcionan al registrarse. Recomendamos probar los endpoint en POSTMAN para hacer las pruebas de la llamada a la API antes de crear el script en Python. Deberíais tener en vuestro proyecto:**`TMDB_API_KEY`** (query `api_key`) o **`TMDB_READ_ACCESS_TOKEN`** (cabecera `Authorization: Bearer …`). Podéis crear un proyecto con variables de entorno o usar una celda de código usando **`getpass`** (entrada oculta, solo esa sesión del kernel). 

No guardéis claves en el notebook

In [8]:
import os
from getpass import getpass

os.environ["TMDB_API_KEY"] = getpass("Inserisci la tua TMDB API Key: ")

In [9]:
import requests

links = pd.read_csv("Team_Challenges/TC_01_Sprint_03_04/data/links.csv")

movies_links = movies.merge(links, on="movieId", how="left")
movies_links = movies_links[movies_links["tmdbId"].notna()]
movies_links["tmdbId"] = movies_links["tmdbId"].astype(int)

movies10 = movies_links.head(10).copy()

def fetch_movie_details(tmdb_id):
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    params = {"api_key": os.getenv("TMDB_API_KEY")}
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return {"overview": "", "homepage": ""}
    data = response.json()
    return {"overview": data.get("overview") or "", "homepage": data.get("homepage") or ""}

movies10["overview"] = ""
movies10["homepage"] = ""
for idx, row in movies10.iterrows():
    details = fetch_movie_details(row["tmdbId"])
    movies10.at[idx, "overview"] = details["overview"]
    movies10.at[idx, "homepage"] = details["homepage"]

movies10[["movieId", "title", "tmdbId", "overview", "homepage"]]

,movieId,title,tmdbId,overview,homepage
0,1,Toy Story,862,"Led by Woody, Andy's toys live happily in his ...",http://toystory.disney.com/toy-story
1,2,Jumanji,8844,When siblings Judy and Peter discover an encha...,http://www.sonypictures.com/movies/jumanji/
2,3,Grumpier Old Men,15602,A family wedding reignites the ancient feud be...,
3,4,Waiting to Exhale,31357,"Cheated on, mistreated and stepped on, the wom...",
4,5,Father of the Bride Part II,11862,Just when George Banks has recovered from his ...,
5,6,Heat,949,Obsessive master thief Neil McCauley leads a t...,https://www.20thcenturystudios.com/movies/heat
6,7,Sabrina,11860,"After her return from school in Paris, a playb...",
7,8,Tom and Huck,45325,"A mischievous young boy, Tom Sawyer, witnesses...",
8,9,Sudden Death,9091,When a man's daughter is suddenly taken during...,
9,10,GoldenEye,710,When a powerful secret defense system is stole...,https://mgm.com/movies/goldeneye


## Parte 3: Sinopsis en español con Gemini

Usad el `DataFrame` **`movies10`** de la Parte 2 (columna `overview` en inglés desde TMDB).

### Tareas

1. Configurar **`GEMINI_API_KEY`** (variable de entorno o `getpass`, como en Sprint 4).
2. Crear el cliente **`google-genai`** y una función **`summarize_overview_es(overview, title="")`** que devuelva un **resumen en español de máximo 2 frases**. Si `overview` está vacío, devolver cadena vacía **sin llamar a la API**.
3. Recorrer `movies10` y añadir la columna **`overview_es`**.
4. Mostrar `title`, `overview` (recorte) y `overview_es` para **3 películas**.

### Autenticación

Clave en [Google AI Studio](https://aistudio.google.com/). Variable **`GEMINI_API_KEY`** o celda con **`getpass`**. No guardéis claves en el notebook.

### Prerrequisitos

- Parte 2 ejecutada (`movies10` con columna `overview`).
- `pip install google-genai`

In [17]:
import os
from getpass import getpass
from google import genai

# Configurazione chiave e client
os.environ["GEMINI_API_KEY"] = getpass("Inserisci la tua GEMINI API Key: ")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-2.5-flash-lite"

def summarize_overview_es(overview, title=""):
    if not overview:
        return ""
    
    prompt = f"""Resume la siguiente sinopsis de película en español, en máximo 2 frases.
Título: {title}
Sinopsis: {overview}

Responde solo con el resumen, sin preámbulos."""
    
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    return response.text.strip()

# Applicare a movies10
movies10["overview_es"] = movies10.apply(
    lambda row: summarize_overview_es(row["overview"], row["title"]), axis=1
)

# Mostrare 3 film
for _, row in movies10.head(3).iterrows():
    print("Título:", row["title"])
    print("Overview (EN):", row["overview"][:150], "...")
    print("Overview (ES):", row["overview_es"])
    print("-" * 80)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## Parte 4 (opcional): extensiones con Gemini

**No obligatorio.** Solo si el grupo terminó las partes 1-3. Podéis elegir **A**, **B** o ambas. Son independientes.

---

### A) Catálogo ampliado en `movies10`

Partiendo de `movies10` (con `overview` y, si ya lo tenéis, `overview_es`), añadid con **una llamada JSON por película**:

- **`pitch_es`**: texto de cartelera en español (máx. 280 caracteres).
- **`edad_sugerida`**: uno de `TP`, `+7`, `+12`, `+16`, `+18`.
- **`temas`**: exactamente 3 temas en español (en el DataFrame, cadena separada por comas).

Función sugerida: **`enrich_catalog_fields(overview, title="", genres="", year=None)`**. Basad la respuesta solo en sinopsis y metadatos; no inventéis reparto ni datos externos. Si `overview` está vacío, no llaméis a la API.

Mostrad 2–3 filas con las columnas nuevas.

---

### B) Recomendador por género

Usad **`ratings_movies`** (Parte 1) y Gemini.

1. **`top_rated_by_genre(ratings_movies, genres, top_n=10, min_ratings=50)`** — filtra por género(s), nota media por `movieId`, devuelve el top N (mínimo `min_ratings` valoraciones por película).
2. **`recommend_movies(candidates, favorite_genres, n=3)`** — el modelo recomienda **solo** del catálogo candidato, con una frase de justificación en español cada una.
3. Probad con 1–2 géneros (p. ej. `["Action", "Sci-Fi"]`): mostrad candidatas y respuesta del modelo.

In [11]:
# --- Parte 4A (opcional): pitch_es, edad_sugerida, temas ---
# Requiere: `movies10` con `overview`; `client` y `MODEL` de la Parte 3.

In [12]:
# --- Parte 4B (opcional): recomendador por género (SOLUCIÓN) ---
# Requiere: `ratings_movies` (Parte 1); `client` y `MODEL` (Parte 3).